# 🚲 Project 4 — Bike Sharing Demand Forecasting

## Recruiter-ready project overview

**Problem type:** Regression / Time Series  
**UCI dataset:** Bike Sharing Dataset (ID 275)

### Executive summary

This project forecasts bike rental demand using historical rental counts, calendar information and weather-related variables. It demonstrates the transition from standard tabular machine learning into **time-aware forecasting**, including datetime feature engineering, cyclical variables, chronological train/test splitting, regression metrics and optional time-series modeling.

### What this project demonstrates

- Datetime parsing and feature engineering
- Hour, weekday, season and weather analysis
- Cyclical encoding with sine/cosine features
- Leakage-aware feature selection
- Chronological train/test splitting
- Linear Regression
- Random Forest Regression
- MAE, RMSE and MAPE
- Actual-vs-predicted analysis
- Feature importance
- Optional SARIMAX forecasting
- Residual analysis

### The key data-science question

> **Can historical calendar and weather information be used to predict future bike rental demand?**

This is a regression problem because the target is a **numeric rental count**.

### Why the chronological split matters

In ordinary tabular classification, random splitting is often useful. For forecasting, randomly mixing future observations into the training set can make performance look unrealistically good. This project therefore respects **time order** when creating the final evaluation set.

### Interview takeaway

A strong explanation is: **“The major lesson was that time-series data requires different validation logic. I engineered calendar and cyclical features, avoided target leakage, trained regression models, and evaluated predictions using MAE, RMSE and MAPE while preserving chronological order.”**


# Project 4 — Bike Sharing Demand Forecasting

**Complexity:** Intermediate–Advanced  
**Problem type:** Regression / Time-Series  
**Dataset:** UCI Bike Sharing Dataset (ID 275)

### What is happening?
This project introduces time-series forecasting into the project sequence. We will predict bike rental demand using weather, time of day, season, and calendar information while respecting the chronological order of the observations.

## 1. Project objective

The goal is to forecast bike rental demand so that operators can make better decisions about bike redistribution, station capacity, and operational planning.

### What is happening?
We are treating rental count as a continuous target. The analysis will focus on how demand changes across hours, seasons, weekdays, holidays, and weather conditions.

## 2. Install and import the required libraries

### What is happening?
This section prepares the Google Colab environment. The `ucimlrepo` package will retrieve the UCI dataset, while pandas, NumPy, Matplotlib, and scikit-learn handle data preparation, visualization, and regression modeling. `statsmodels` is used later for an optional time-series model.

In [ ]:
!pip -q install ucimlrepo statsmodels

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ucimlrepo import fetch_ucirepo

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

from statsmodels.tsa.statespace.sarimax import SARIMAX

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

## 3. Load the UCI Bike Sharing dataset

### What is happening?
The proposal specifies UCI dataset **ID 275**. We load the data directly from UCI and inspect the feature and target tables before combining them into one working DataFrame.

In [ ]:
bike = fetch_ucirepo(id=275)

X_raw = bike.data.features.copy()
y_raw = bike.data.targets.copy()

print("Feature shape:", X_raw.shape)
print("Target shape:", y_raw.shape)

print("\nFeature columns:")
print(X_raw.columns.tolist())

print("\nTarget columns:")
print(y_raw.columns.tolist())

display(X_raw.head())
display(y_raw.head())

## 4. Build the working dataset

### What is happening?
The UCI package normally returns predictors and target information separately. We combine them, identify the rental-count target, and make the column names easier to work with.

In [ ]:
df = X_raw.copy()

if "cnt" in y_raw.columns:
    df["cnt"] = y_raw["cnt"].values
elif "count" in y_raw.columns:
    df["cnt"] = y_raw["count"].values
elif y_raw.shape[1] == 1:
    df["cnt"] = y_raw.iloc[:, 0].values
else:
    raise ValueError("Could not identify the bike rental count target.")

print("Working shape:", df.shape)
display(df.head())

## 5. Inspect the structure and data quality

### What is happening?
Before modeling, we check data types, missing values, duplicates, and basic statistics. This helps identify formatting problems before feature engineering.

In [ ]:
print("Shape:", df.shape)
print("\nData types:")
display(df.dtypes)

print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False).head(20))

print("\nDuplicate rows:", df.duplicated().sum())

print("\nDescriptive statistics:")
display(df.describe(include="all").T)

## 6. Preserve an untouched raw copy

### What is happening?
A raw copy gives us a reproducible reference point. All cleaning and feature engineering will be performed on the working copy rather than overwriting the original imported data.

In [ ]:
df_raw = df.copy()
df = df.copy()

print("Raw copy preserved:", df_raw.shape)

## 7. Convert the date and time fields

### What is happening?
The proposal requires a proper datetime index. The hourly dataset stores a date (`dteday`) and hour (`hr`), so we combine them into a single timestamp. This is essential for chronological analysis.

In [ ]:
df["dteday"] = pd.to_datetime(df["dteday"], errors="coerce")

if "hr" in df.columns:
    df["datetime"] = df["dteday"] + pd.to_timedelta(df["hr"], unit="h")
else:
    df["datetime"] = df["dteday"]

df = df.sort_values("datetime").reset_index(drop=True)

print("Datetime range:")
print(df["datetime"].min(), "to", df["datetime"].max())

display(df[["dteday", "hr", "datetime", "cnt"]].head())

## 8. Basic cleaning

### What is happening?
Rows without a valid timestamp or target cannot be used for time-series modeling. We remove those rows, remove exact duplicates, and check the resulting dataset.

In [ ]:
df = df.dropna(subset=["datetime", "cnt"]).drop_duplicates().reset_index(drop=True)

print("Shape after cleaning:", df.shape)
print("Missing datetime:", df["datetime"].isna().sum())
print("Missing target:", df["cnt"].isna().sum())

## 9. Create time-based features

### What is happening?
Time-series demand often follows repeating cycles. We extract hour, day of week, month, year, weekend status, and other calendar features so that regression models can learn these patterns.

In [ ]:
df["year"] = df["datetime"].dt.year
df["month"] = df["datetime"].dt.month
df["day"] = df["datetime"].dt.day
df["day_of_week"] = df["datetime"].dt.dayofweek
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

if "hr" not in df.columns:
    df["hour"] = df["datetime"].dt.hour
else:
    df["hour"] = df["hr"].astype(int)

display(df[["datetime", "hour", "day_of_week", "month", "year", "is_weekend"]].head())

## 10. Add cyclical time encoding

### What is happening?
Hour 23 and hour 0 are close in real time, but ordinary numeric encoding treats them as far apart. Sine/cosine features represent these repeating cycles smoothly, following the proposal's requirement for cyclical time features.

In [ ]:
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

display(df[[
    "hour", "hour_sin", "hour_cos",
    "day_of_week", "dow_sin", "dow_cos",
    "month", "month_sin", "month_cos"
]].head())

## 11. Inspect the weather variables

### What is happening?
The dataset contains weather-related variables such as temperature, humidity, windspeed, and weather situation. We check their ranges and category values before visualization and modeling.

In [ ]:
weather_cols = [c for c in ["temp", "atemp", "hum", "windspeed", "weathersit"] if c in df.columns]

for col in weather_cols:
    print(f"\n{col}")
    print(df[col].describe())
    if df[col].nunique() <= 10:
        print("Values:", sorted(df[col].dropna().unique().tolist()))

## 12. Explore overall rental demand

### What is happening?
We first understand the target itself. The histogram shows the distribution of hourly rental counts and helps us see whether demand is concentrated at low or high values.

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(df["cnt"], bins=50)
plt.xlabel("Bike rentals")
plt.ylabel("Frequency")
plt.title("Distribution of Bike Rental Demand")
plt.show()

## 13. Hourly demand pattern

### What is happening?
We calculate average demand by hour. This reveals daily usage patterns such as commuting peaks and quieter overnight periods.

In [ ]:
hourly_demand = df.groupby("hour")["cnt"].mean()

plt.figure(figsize=(11, 5))
plt.plot(hourly_demand.index, hourly_demand.values, marker="o")
plt.xlabel("Hour of day")
plt.ylabel("Average rentals")
plt.title("Average Bike Demand by Hour")
plt.xticks(range(0, 24))
plt.grid(True, alpha=0.3)
plt.show()

## 14. Seasonal demand pattern

### What is happening?
We compare average rentals across seasons. This shows how demand changes under different seasonal conditions.

In [ ]:
season_demand = df.groupby("season")["cnt"].mean()

plt.figure(figsize=(8, 5))
plt.bar(season_demand.index.astype(str), season_demand.values)
plt.xlabel("Season code")
plt.ylabel("Average rentals")
plt.title("Average Bike Demand by Season")
plt.show()

display(season_demand.to_frame("average_rentals"))

## 15. Weekday versus weekend demand

### What is happening?
This comparison tests whether demand behaves differently on working days and weekends, which is useful for operational scheduling.

In [ ]:
weekday_demand = df.groupby("is_weekend")["cnt"].mean()

labels = ["Weekday", "Weekend"]

plt.figure(figsize=(7, 5))
plt.bar(labels, [weekday_demand.get(0, np.nan), weekday_demand.get(1, np.nan)])
plt.ylabel("Average rentals")
plt.title("Weekday vs Weekend Bike Demand")
plt.show()

## 16. Holiday and working-day demand

### What is happening?
Holiday status and working-day status can capture changes in commuting behavior. We compare average rental counts across these calendar conditions.

In [ ]:
fig_data = []

if "holiday" in df.columns:
    fig_data.append(("holiday", df.groupby("holiday")["cnt"].mean()))

if "workingday" in df.columns:
    fig_data.append(("workingday", df.groupby("workingday")["cnt"].mean()))

for name, series in fig_data:
    plt.figure(figsize=(7, 5))
    plt.bar(series.index.astype(str), series.values)
    plt.xlabel(name)
    plt.ylabel("Average rentals")
    plt.title(f"Average Demand by {name.title()}")
    plt.show()

    display(series.to_frame("average_rentals"))

## 17. Weather impact visualizations

### What is happening?
The proposal specifically asks for weather-impact plots. Scatter plots let us inspect how temperature, humidity, and windspeed relate to rental demand.

In [ ]:
for col in ["temp", "hum", "windspeed"]:
    if col in df.columns:
        plt.figure(figsize=(8, 5))
        plt.scatter(df[col], df["cnt"], alpha=0.15)
        plt.xlabel(col)
        plt.ylabel("Bike rentals")
        plt.title(f"{col} vs Bike Rental Demand")
        plt.grid(True, alpha=0.2)
        plt.show()

## 18. Demand over time

### What is happening?
Plotting the target against the chronological timestamp gives a direct view of seasonality, long-term growth, unusual peaks, and changing demand levels.

In [ ]:
daily_demand = df.set_index("datetime")["cnt"].resample("D").sum()

plt.figure(figsize=(14, 5))
plt.plot(daily_demand.index, daily_demand.values)
plt.xlabel("Date")
plt.ylabel("Daily rentals")
plt.title("Daily Bike Rental Demand Over Time")
plt.grid(True, alpha=0.2)
plt.show()

## 19. Prepare the modeling dataset

### What is happening?
We remove variables that would leak the answer into the model. In particular, `casual` and `registered` are components of total count and therefore must not be used to predict `cnt`. We also avoid the raw timestamp and redundant identifiers.

In [ ]:
df_model = df.copy()

leakage_or_identifier_cols = [
    "cnt", "casual", "registered", "instant", "dteday", "datetime"
]

leakage_or_identifier_cols = [
    c for c in leakage_or_identifier_cols if c in df_model.columns
]

X = df_model.drop(columns=leakage_or_identifier_cols)
y = df_model["cnt"].astype(float)

print("Predictor columns:")
print(X.columns.tolist())
print("\nTarget:", y.name)

## 20. Use a chronological train/test split

### What is happening?
A random split would allow future observations to influence training. Because this is time-series data, we train on earlier observations and test on later observations, as required by the proposal.

In [ ]:
split_index = int(len(df_model) * 0.80)

X_train = X.iloc[:split_index].copy()
X_test = X.iloc[split_index:].copy()

y_train = y.iloc[:split_index].copy()
y_test = y.iloc[split_index:].copy()

print("Training period:", df_model["datetime"].iloc[0], "to", df_model["datetime"].iloc[split_index - 1])
print("Testing period:", df_model["datetime"].iloc[split_index], "to", df_model["datetime"].iloc[-1])
print("Train rows:", len(X_train))
print("Test rows:", len(X_test))

## 21. Identify numerical and categorical predictors

### What is happening?
The preprocessing pipeline will treat numeric and categorical variables differently. Numeric values are imputed and standardized for Linear Regression, while categorical variables are one-hot encoded.

In [ ]:
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

## 22. Build the preprocessing pipeline

### What is happening?
Using a pipeline prevents preprocessing mistakes and ensures the same transformations are applied to training and test data.

In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

## 23. Define evaluation metrics

### What is happening?
The proposal specifies RMSE, MAE, and MAPE. MAE gives the average absolute error in rental units, RMSE penalizes larger errors more heavily, and MAPE expresses error as a percentage. Zero actual values are excluded from MAPE to avoid division by zero.

In [ ]:
def regression_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    y_true_array = np.asarray(y_true)
    y_pred_array = np.asarray(y_pred)
    nonzero = y_true_array != 0

    if nonzero.sum() > 0:
        mape = np.mean(
            np.abs((y_true_array[nonzero] - y_pred_array[nonzero]) /
                   y_true_array[nonzero])
        ) * 100
    else:
        mape = np.nan

    return {
        "MAE": mae,
        "RMSE": rmse,
        "MAPE (%)": mape
    }

## 24. Model 1 — Linear Regression baseline

### What is happening?
Linear Regression provides an interpretable baseline. It tests how well demand can be explained through additive linear relationships between the engineered calendar, weather, and seasonal predictors.

In [ ]:
linear_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

linear_model.fit(X_train, y_train)
linear_pred = linear_model.predict(X_test)

linear_results = regression_metrics(y_test, linear_pred)
linear_results

## 25. Model 2 — Random Forest Regressor

### What is happening?
Random Forest can capture non-linear relationships and interactions that Linear Regression may miss. This makes it a useful comparison for demand patterns influenced by combinations of time and weather.

In [ ]:
rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
        max_depth=None
    ))
])

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

rf_results = regression_metrics(y_test, rf_pred)
rf_results

## 26. Compare the regression models

### What is happening?
We place the metrics side by side. Lower MAE, RMSE, and MAPE indicate better forecasting performance.

In [ ]:
results = pd.DataFrame({
    "Linear Regression": linear_results,
    "Random Forest": rf_results
}).T

display(results.sort_values("RMSE"))

## 27. Visualize actual versus predicted demand

### What is happening?
A metric alone does not show how a model behaves over time. This plot compares actual and predicted rental counts during the chronological test period.

In [ ]:
test_plot = df_model.iloc[split_index:][["datetime"]].copy()
test_plot["actual"] = y_test.values
test_plot["linear_pred"] = linear_pred
test_plot["rf_pred"] = rf_pred

plt.figure(figsize=(15, 6))
plt.plot(test_plot["datetime"], test_plot["actual"], label="Actual", alpha=0.7)
plt.plot(test_plot["datetime"], test_plot["linear_pred"], label="Linear Regression", alpha=0.7)
plt.plot(test_plot["datetime"], test_plot["rf_pred"], label="Random Forest", alpha=0.7)
plt.xlabel("Date")
plt.ylabel("Bike rentals")
plt.title("Actual vs Predicted Bike Demand")
plt.legend()
plt.grid(True, alpha=0.2)
plt.show()

## 28. Inspect a shorter forecasting window

### What is happening?
The full test period can be visually crowded. A shorter window makes it easier to inspect whether the model follows individual demand peaks and troughs.

In [ ]:
window = min(24 * 7, len(test_plot))

plt.figure(figsize=(15, 6))
plt.plot(test_plot["datetime"].iloc[:window], test_plot["actual"].iloc[:window], label="Actual")
plt.plot(test_plot["datetime"].iloc[:window], test_plot["rf_pred"].iloc[:window], label="Random Forest")
plt.xlabel("Date")
plt.ylabel("Bike rentals")
plt.title("One-Week Actual vs Random Forest Forecast")
plt.legend()
plt.grid(True, alpha=0.2)
plt.show()

## 29. Random Forest feature importance

### What is happening?
The proposal asks for model interpretation. We extract the transformed feature names and rank them by Random Forest importance to identify which time, weather, and calendar signals contribute most to predictions.

In [ ]:
rf_preprocessor = rf_model.named_steps["preprocessor"]
rf_estimator = rf_model.named_steps["model"]

feature_names = rf_preprocessor.get_feature_names_out()
importances = rf_estimator.feature_importances_

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

display(importance_df.head(20))

## 30. Plot the most important predictors

### What is happening?
This chart converts the feature-importance output into an easier visual summary of the strongest predictors used by the Random Forest.

In [ ]:
top_n = 15
top_features = importance_df.head(top_n).sort_values("importance")

plt.figure(figsize=(10, 7))
plt.barh(top_features["feature"], top_features["importance"])
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top Random Forest Predictors of Bike Demand")
plt.show()

## 31. Optional time-series model — SARIMAX

### What is happening?
The proposal allows a time-series model in addition to the regression models. Here we demonstrate SARIMAX on **daily total demand**. This section is intentionally kept separate from the machine-learning pipeline because classical time-series models use the temporal sequence directly.

For a faster Colab run, the example uses a smaller recent daily window. You can increase the window after confirming that it runs successfully.

In [ ]:
daily = (
    df_model[["datetime", "cnt"]]
    .set_index("datetime")
    .resample("D")
    .sum()
    .asfreq("D")
)

daily["cnt"] = daily["cnt"].interpolate()

sarimax_test_size = min(60, max(14, len(daily) // 5))

sarimax_train = daily.iloc[:-sarimax_test_size]["cnt"]
sarimax_test = daily.iloc[-sarimax_test_size:]["cnt"]

print("SARIMAX train:", sarimax_train.index.min(), "to", sarimax_train.index.max())
print("SARIMAX test:", sarimax_test.index.min(), "to", sarimax_test.index.max())

## 32. Fit and evaluate SARIMAX

### What is happening?
We fit a simple seasonal SARIMAX specification using daily demand. The goal is to provide a classical time-series benchmark rather than to claim that this configuration is automatically optimal.

In [ ]:
sarimax_model = SARIMAX(
    sarimax_train,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False
)

sarimax_fit = sarimax_model.fit(disp=False)

sarimax_forecast = sarimax_fit.forecast(steps=len(sarimax_test))
sarimax_results = regression_metrics(sarimax_test, sarimax_forecast)

print("SARIMAX results:")
display(pd.DataFrame([sarimax_results], index=["SARIMAX"]))

## 33. Visualize the SARIMAX forecast

### What is happening?
This plot checks whether the time-series model follows the daily demand pattern during its holdout period.

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(sarimax_test.index, sarimax_test.values, label="Actual")
plt.plot(sarimax_test.index, sarimax_forecast.values, label="SARIMAX forecast")
plt.xlabel("Date")
plt.ylabel("Daily rentals")
plt.title("SARIMAX Daily Demand Forecast")
plt.legend()
plt.grid(True, alpha=0.2)
plt.show()

## 34. Compare regression and time-series performance

### What is happening?
The regression models predict hourly observations, while SARIMAX above predicts daily totals. Because the targets are at different time resolutions, the metrics should not be treated as a perfectly apples-to-apples ranking. The comparison is mainly useful for understanding the strengths of the two modeling approaches.

In [ ]:
comparison = pd.concat([
    results,
    pd.DataFrame([sarimax_results], index=["SARIMAX"])
], axis=0)

display(comparison)

## 35. Check model residuals

### What is happening?
Residuals are the difference between actual and predicted demand. Looking at their distribution can reveal systematic underprediction, overprediction, or unusually large errors.

In [ ]:
residuals = y_test.values - rf_pred

plt.figure(figsize=(10, 5))
plt.hist(residuals, bins=50)
plt.xlabel("Actual - Predicted")
plt.ylabel("Frequency")
plt.title("Random Forest Forecast Residuals")
plt.show()

print("Mean residual:", residuals.mean())
print("Median residual:", np.median(residuals))

## 36. Create a final model summary table

### What is happening?
This section creates a clean table that can be copied into a project report. It records the main regression metrics for the machine-learning models.

In [ ]:
final_ml_results = results.copy()
final_ml_results["Model"] = final_ml_results.index
final_ml_results = final_ml_results[["Model", "MAE", "RMSE", "MAPE (%)"]]

display(final_ml_results.sort_values("RMSE"))

## 37. Save the cleaned dataset and model outputs

### What is happening?
The proposal requires a reproducible project. We save the cleaned time-indexed data, model comparison metrics, and feature-importance results so they can be reused in a report or later analysis.

In [ ]:
output_dir = "bike_sharing_outputs"
import os
os.makedirs(output_dir, exist_ok=True)

df.to_csv(f"{output_dir}/bike_sharing_cleaned_time_indexed.csv", index=False)
final_ml_results.to_csv(f"{output_dir}/model_comparison.csv", index=False)
importance_df.to_csv(f"{output_dir}/random_forest_feature_importance.csv", index=False)

print("Saved files:")
print(os.listdir(output_dir))

## 38. Project interpretation

### What is happening?
Now we turn the numerical results into a practical story. Do not invent findings: replace the placeholders below with the actual patterns observed in your notebook outputs.

### Questions to answer
- Which hours have the highest demand?
- How does demand differ between weekdays and weekends?
- Which seasons show the strongest rental activity?
- How does weather affect demand?
- Which model has the lowest MAE/RMSE/MAPE?
- Which predictors are most important?
- Where does the model make its largest errors?

## 39. Sustainable city-planning recommendations

### What is happening?
The proposal asks for a short brief connecting demand patterns to sustainable city planning. The recommendations should be based on the evidence produced above.

Possible evidence-backed areas include:
- reallocating bikes toward high-demand periods and locations;
- increasing station capacity where sustained demand is observed;
- preparing additional operational capacity around seasonal or weather-related peaks;
- improving availability during recurring commuter peaks;
- using forecasts to reduce unnecessary redistribution trips and improve operational efficiency.

Only retain recommendations that are supported by your actual results.

## 40. Final conclusion template

### What is happening?
This final section gives you a structure for the written conclusion. Replace the bracketed parts with your actual results after running all cells.

**Conclusion**

This project forecasted bike rental demand using the UCI Bike Sharing dataset. The analysis transformed the original date and hour information into chronological and cyclical time features and incorporated seasonal, calendar, and weather variables.

The exploratory analysis showed that demand varied by **[hour/season/day type/weather condition]**. Among the tested machine-learning models, **[best model]** achieved the strongest performance with an RMSE of **[value]**, MAE of **[value]**, and MAPE of **[value]%**.

The feature-importance analysis indicated that **[top predictors]** were among the strongest signals for demand. These findings can support operational decisions such as **[bike redistribution/station capacity/planning]** and contribute to more efficient and sustainable urban mobility planning.

### Limitations
- The regression models rely on the available weather and calendar variables.
- A chronological holdout gives a realistic test but may still differ from future operating conditions.
- The SARIMAX demonstration uses a simple specification and should be tuned further for a production forecasting system.
- Forecast accuracy can deteriorate when unusual events or conditions are not represented in the training data.

# Project 4 checklist

### What is happening?
Use this checklist before considering Project 4 complete.

- [ ] UCI dataset ID 275 loaded successfully.
- [ ] Raw dataset preserved.
- [ ] Datetime created and sorted chronologically.
- [ ] Hour/day/month cyclical features created.
- [ ] Weather variables inspected.
- [ ] Seasonal and hourly demand visualizations completed.
- [ ] Weather-impact plots completed.
- [ ] Weekday/weekend and holiday comparisons completed.
- [ ] Chronological train/test split used.
- [ ] Linear Regression trained.
- [ ] Random Forest Regressor trained.
- [ ] Optional SARIMAX benchmark completed.
- [ ] MAE, RMSE, and MAPE calculated.
- [ ] Actual vs predicted demand plotted.
- [ ] Feature importance interpreted.
- [ ] Cleaned dataset and model outputs saved.
- [ ] Final storytelling and sustainable-planning recommendations written.

## 🎯 Portfolio conclusion — Project 4

This project demonstrates the move from ordinary tabular ML into time-aware forecasting.

### Knowledge checkpoints

- **Regression:** predict a numeric value.
- **Time series:** observations indexed by time.
- **Chronological split:** train on earlier observations and test on later observations.
- **Cyclical feature:** represent repeating time patterns with sine/cosine.
- **MAE:** average absolute prediction error.
- **RMSE:** penalizes larger errors more strongly.
- **MAPE:** expresses average error relative to actual values.

### What to remember

For forecasting, validation must respect time. A model should not learn from information that would only be available after the prediction point.
